In [ ]:
#SIMPLER ANGLES
from AIce.functions import torchRMSE,trainloader,testloader,redim,getRMSE
from torch import optim
from AIce.models import simplrNNforAngles,simplrNNforNorms
import numpy as np
import torch
import matplotlib.pyplot as plt

from time import strftime


device='cuda'

inputlist=['u_ERA5','v_ERA5','h_piomas','bath', 'windnorm']
lr =1e-3
n_epoch=20
data,_,dataloader,means,stds,maxes,labels=trainloader('../data/DRIFT_DATA_TRAIN.csv',32,inputlist,target='angle')
print(labels)
# testdata,_,_,means,stds,maxes,labels=trainloader('../data/DRIFT_DATA_TRAIN.csv',256,inputlist=inputlist ,target='buoynorm')

# data['buoynorm']=testdata['buoynorm']

print(labels)
print('aaa')
print(labels[1])

mlp256sa=simplrNNforAngles(len(labels[1])).to(device)


print('Total Epoch: ', n_epoch)
print('Learning Rate: ', lr)
print(labels)

rlosses={}
losses=[]
rloss=0
rcount=0

optimizer = optim.Adam(mlp256sa.parameters(),lr=lr) # optimizers

for epoch in range(n_epoch):
    print('Starting Epoch ',epoch)
    curpercent=0
    rl=[]
    for i,data in enumerate(dataloader):

        # get the inputs and the target
        truth,inputs= data[0].to(device),data[1].to(device)

        # zero the gradients,
        optimizer.zero_grad()

        # Forward,
        out=mlp256sa(inputs)


        loss = torchRMSE(target=truth, outputs=out)
        # Backward
        loss.backward()
        losses.append(loss.item())
        # Optimize

        optimizer.step()

        rloss+= loss.item()
        rcount +=1
        #print where we at plus the loss
        percent= round(i/len(dataloader)*100)
        if percent != curpercent:
        #    print(percent, 'loss: ', rloss/rcount)
            curpercent=percent
            rl.append(rloss/rcount)
            rloss=0
            rcount=0
    rlosses[epoch]=rl
    print('Epoch avg loss: ', np.mean(rl))

saving_name=f'simplr_angles_{len(labels[1])}inputs_{n_epoch}E_{lr}lr_{strftime("%d-%Hh_%Mm_%Ss")}'

torch.save(mlp256sa.state_dict(), f'.//weights//{saving_name}.pt')

with open(f'.//weights//{saving_name}.txt','w') as f:
    f.write(f'Model: simplrNNforAngle (on gpu)\n')
    f.write(f'Weights: {saving_name}.pt\n')
    f.write(f'Associated inputs: {inputlist}')

fig,ax =plt.subplots()
epochavg=[]

for i in (rlosses):
    x=np.arange(i*len(rlosses[0]),len(rlosses[0])+i*len(rlosses[0]))
    ax.plot(x,rlosses[i])#,label=f'Epoch {i+1}, lr={lr[i]:.1e}')
    avg=np.mean(rlosses[i])
    epochavg.append(avg)
xx=[i*len(rlosses[0]) for i in rlosses]
ax.plot(xx,epochavg,'.-k')
plt.savefig(f'.//outputs//loss_for_{saving_name}.png')
plt.close('all')


static cases removed for training
allright angles
Bathymetry (bath) normalized by maximum
Windnorm normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
[Index(['angle'], dtype='object'), Index(['u_ERA5', 'v_ERA5', 'h_piomas', 'bath', 'windnorm'], dtype='object')]
[Index(['angle'], dtype='object'), Index(['u_ERA5', 'v_ERA5', 'h_piomas', 'bath', 'windnorm'], dtype='object')]
aaa
Index(['u_ERA5', 'v_ERA5', 'h_piomas', 'bath', 'windnorm'], dtype='object')


TypeError: Module.get_parameter() missing 1 required positional argument: 'target'

In [3]:
#SIMPLER NORMS
from AIce.functions import torchRMSE,trainloader,testloader,redim,getRMSE
from torch import optim
from AIce.models import simplrNNforAngles,simplrNNforNorms
import numpy as np
import torch
import matplotlib.pyplot as plt

from time import strftime


device='cuda'

inputlist=['u_ERA5','v_ERA5','h_piomas','bath', 'windnorm']
lr =1e-3
n_epoch=20
data,_,dataloader,means,stds,maxes,labels=trainloader('../data/DRIFT_DATA_TRAIN.csv',32,inputlist,target='buoynorm')
# print(labels)
# testdata,_,_,means,stds,maxes,labels=trainloader('../data/DRIFT_DATA_TRAIN.csv',256,inputlist=inputlist ,target='buoynorm')

# data['buoynorm']=testdata['buoynorm']

print(labels)

print(labels[1])

mlp256sn=simplrNNforNorms(len(labels[1])).to(device)
print('Total Epoch: ', n_epoch)
print('Learning Rate: ', lr)
print(labels)

rlosses={}
losses=[]
rloss=0
rcount=0

optimizer = optim.Adam(mlp256sn.parameters(),lr=lr) # optimizers

for epoch in range(n_epoch):
    print('Starting Epoch ',epoch)
    curpercent=0
    rl=[]
    for i,data in enumerate(dataloader):

        # get the inputs and the target
        truth,inputs= data[0].to(device),data[1].to(device)

        # zero the gradients,
        optimizer.zero_grad()

        # Forward,
        out=mlp256sn(inputs)


        loss = torchRMSE(target=truth, outputs=out)
        # Backward
        loss.backward()
        losses.append(loss.item())
        # Optimize

        optimizer.step()

        rloss+= loss.item()
        rcount +=1
        #print where we at plus the loss
        percent= round(i/len(dataloader)*100)
        if percent != curpercent:
        #    print(percent, 'loss: ', rloss/rcount)
            curpercent=percent
            rl.append(rloss/rcount)
            rloss=0
            rcount=0
    rlosses[epoch]=rl
    print('Epoch avg loss: ', np.mean(rl))

saving_name=f'simplr_norms_{len(labels[1])}inputs_{n_epoch}E_{lr}lr_{strftime("%d-%Hh_%Mm_%Ss")}'

torch.save(mlp256sn.state_dict(), f'.//weights//{saving_name}.pt')

with open(f'.//weights//{saving_name}.txt','w') as f:
    f.write(f'Model: simplrNNforNorms (on gpu)\n')
    f.write(f'Weights: {saving_name}.pt\n')
    f.write(f'Associated inputs: {inputlist}')

fig,ax =plt.subplots()
epochavg=[]

for i in (rlosses):
    x=np.arange(i*len(rlosses[0]),len(rlosses[0])+i*len(rlosses[0]))
    ax.plot(x,rlosses[i])#,label=f'Epoch {i+1}, lr={lr[i]:.1e}')
    avg=np.mean(rlosses[i])
    epochavg.append(avg)
xx=[i*len(rlosses[0]) for i in rlosses]
ax.plot(xx,epochavg,'.-k')
plt.savefig(f'.//outputs//loss_for_{saving_name}.png')
plt.close('all')


Target is buoynorm normalized by log1p
Bathymetry (bath) normalized by maximum
Windnorm normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
[Index(['buoynorm'], dtype='object'), Index(['u_ERA5', 'v_ERA5', 'h_piomas', 'bath', 'windnorm'], dtype='object')]
Index(['u_ERA5', 'v_ERA5', 'h_piomas', 'bath', 'windnorm'], dtype='object')
Total Epoch:  20
Learning Rate:  0.001
[Index(['buoynorm'], dtype='object'), Index(['u_ERA5', 'v_ERA5', 'h_piomas', 'bath', 'windnorm'], dtype='object')]
Starting Epoch  0


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import datetime
from scipy.interpolate import griddata

from AIce.functions import trainloader,redim,getRMSE,torchRMSE
from AIce.models import simplrNNforNorms,simplrNNforAngles


import cartopy.crs as ccrs



In [16]:
device='cuda'
allinput=['u_ERA5','v_ERA5','h_piomas','sic_CDR','x_EASE','y_EASE','bath','sin','cos', 'windnorm']
#lst=['u_ERA5', 'v_ERA5', 'h_piomas', 'sic_CDR', 'bath']
#weightpath='./weights/UV_5inputs_20E_0.001lr_18-13h_37m_54s.pt'

testdata,_,_,means,stds,maxes,labels=trainloader('../data/DRIFT_DATA_TRAIN.csv',inputlist=allinput ,target='buoynorm', bs=256)
                                # trainingset_loaded=False, training_file='../data/DRIFT_DATA_TRAIN.csv')# Get the means,stds,maxes using the longest list possible

testdata=redim(testdata,
               means=means,
               stds=stds,
               maxes=maxes)



testdata['anglewind']=np.degrees(np.arctan2(testdata['u_ERA5'],testdata['v_ERA5']))
testdata1,_,_,means1,stds1,maxes1,labels1=trainloader('../data/DRIFT_DATA_TRAIN.csv',inputlist=allinput ,target='angle', bs=256)
                               # trainingset_loaded=False, training_file='../data/DRIFT_DATA_TRAIN.csv')# Get the means,stds,maxes using the longest list possible

testdata1=redim(testdata1,
               means=means1,
               stds=stds1,
               maxes=maxes1)

testdata['angle']=testdata1['angle']
print(len(testdata), testdata.columns)

Target is buoynorm normalized by log1p
Sin and Cos added
Bathymetry (bath) normalized by maximum
x/y (x_EASE, y_EASE) normalized by maximum
Windnorm normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
static cases removed for training
allright angles
Sin and Cos added
Bathymetry (bath) normalized by maximum
x/y (x_EASE, y_EASE) normalized by maximum
Windnorm normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
339156 Index(['buoynorm', 'x_EASE', 'y_EASE', 'u_ERA5', 'v_ERA5', 'sic_CDR',
       'h_piomas', 'sin', 'cos', 'bath', 'windnorm', 'anglewind', 'angle'],
      dtype='object')
